# Notebook 4 — Feature Output Contracts: Result Dataclasses and `to_flat_dict()`

> **Learning Objectives**
> - Understand why PyEyesWeb returns typed dataclasses instead of raw floats
> - Inspect `FeatureResult` and its specialisations: `KineticEnergyResult`, `SmoothnessResult`, `ContractionExpansionResult`, `RarityResult`, `SuddennessResult`
> - Use `is_valid` to detect computation failures gracefully
> - Use `to_flat_dict(prefix=...)` to build a clean Pandas DataFrame logging pipeline
> - Export a multi-feature time-series table to CSV

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pyeyesweb.data_models import SlidingWindow
from pyeyesweb.low_level import KineticEnergy, PointsDensity, Smoothness, DirectionChange
from pyeyesweb.mid_level import Suddenness
from pyeyesweb.analysis_primitives import Rarity
from utils.data_loader import GestureDataLoader

loader = GestureDataLoader("data")

## 1. The Problem with Raw Floats

Imagine a feature that computes three things simultaneously: total kinetic energy, per-axis energy components, and per-joint energies. If the return type were a plain Python list, the caller would have to know (and remember) that:

- Index `0` is total energy
- Indices `1:4` are X/Y/Z components
- Indices `4:` are per-joint values

This is fragile, undocumented, and error-prone — especially when refactoring or routing data over a network (e.g., OSC).

**PyEyesWeb's solution**: every feature returns a strictly typed Python `@dataclass` that inherits from `FeatureResult`. Fields have names, types, and docstrings. Autocomplete works. Bugs are caught at definition time, not runtime.

---

## 2. The `FeatureResult` Base Class

In [2]:
from pyeyesweb.data_models.results import FeatureResult

All result classes share two guarantees:

| Attribute / Method | Purpose |
|-------------------|---------|
| `is_valid: bool` | `True` if computation succeeded; `False` on failure (empty window, too few samples, degenerate geometry) |
| `to_flat_dict(prefix="")` | Flattens all fields to a dictionary of scalar values, optionally namespaced |

### 2.1 The `is_valid` contract — graceful failure

When a feature cannot produce a meaningful result, it returns a `FeatureResult` with `is_valid=False` rather than raising an exception. This is particularly important in real-time loops where a single bad frame should not crash the system.

In [3]:
from pyeyesweb.data_models.results import FeatureResult

empty_window = SlidingWindow(max_length=60, n_signals=1, n_dims=1)
smooth_ft    = Smoothness(rate_hz=100.0)

result = smooth_ft(empty_window)   # Window is empty!
print(f"is_valid: {result.is_valid}")   # → False
print(f"Jerk: {getattr(result, "Jerk", None)}")          # → None (optional field, not computed)

is_valid: False
Jerk: None


In a real-time loop, you can use this cleanly:
```python
result = smooth_ft(window)
if result.is_valid:
    send_over_osc("/smoothness", result.sparc)
```

## 3. Anatomy of Specific Result Types

### 3.1 `KineticEnergyResult`

In [14]:
pos_tensor, vel_tensor, _, marker_names = loader.load("trial20", sensor="qualisys")
N_frames, N_joints, N_dims = pos_tensor.shape

ef = KineticEnergy(weights=1.0, labels=marker_names)
sw = SlidingWindow(max_length=1, n_signals=N_joints, n_dims=3)
sw.append(vel_tensor[100, :, :])  # Append a single frame

result = ef(sw)
print(result)

KineticEnergyResult(is_valid=True, total_energy=55657.78520205262, component_energy=[51416.605165607965, 721.3199958021942, 3519.8600406424725], joints={'HeadFront': {'total': 1552.3849999809263, 'components': [1540.125, 5.780000324249272, 6.479999656677251]}, 'HeadL': {'total': 1885.289913187028, 'components': [1806.004908294679, 12.00500046730042, 67.2800044250489]}, 'HeadR': {'total': 1934.920047478676, 'components': [1934.4200474548343, 0.18000001430511503, 0.32000000953674324]}, 'Chest': {'total': 1340.9449204015743, 'components': [1331.2799212646496, 8.819999198913592, 0.8449999380111706]}, 'LShoulderTop': {'total': 1674.4000442504885, 'components': [1635.920043640137, 20.480000610351567, 18.0]}, 'RShoulderTop': {'total': 1636.6899553871158, 'components': [1613.1199566650394, 1.125, 22.444998722076434]}, 'SpineThoracic2': {'total': 1828.449999666214, 'components': [1800.0, 17.40500056266785, 11.04499910354616]}, 'SpineThoracic12': {'total': 1652.8900438165667, 'components': [1635

**Output (condensed):**
```
KineticEnergyResult(
    is_valid=True,
    total_energy=0.423,
    component_energy=[0.102, 0.198, 0.123],   # X, Y, Z
    joints={
        "PELVIS":     {"total": 0.021, "components": [0.005, 0.010, 0.006]},
        "HAND_RIGHT": {"total": 0.054, "components": [0.019, 0.024, 0.011]},
        ...
    }
)
```

Accessing individual fields:

In [15]:
print(f"Total energy     : {result.total_energy:.4f}")
print(f"Component X      : {result.component_energy[0]:.4f}")
print(f"HAND_RIGHT total : {result.joints['LWristOut']['total']:.4f}")

Total energy     : 55657.7852
Component X      : 51416.6052
HAND_RIGHT total : 2430.0151


### 3.2 `SmoothnessResult`

In [16]:
sf = Smoothness(rate_hz=100.0, metrics=["sparc", "jerk_rms"])
sw_speed = SlidingWindow(max_length=60, n_signals=1, n_dims=1)
hand_idx = marker_names.index("LWristOut")
hand_speed = np.linalg.norm(vel_tensor[:, hand_idx, :], axis=1)

for t in range(60):
    sw_speed.append(hand_speed[t])

result = sf(sw_speed)
print(result)
# → SmoothnessResult(is_valid=True, sparc=-2.341, jerk_rms=0.187)
print(f"SPARC    : {result.sparc:.3f}")
print(f"Jerk RMS : {result.jerk_rms:.3f}")

SmoothnessResult(is_valid=True, sparc=-2.6053711825404355, jerk_rms=21949.07421875)
SPARC    : -2.605
Jerk RMS : 21949.074


### 3.3 `ContractionExpansionResult`

In [17]:
from pyeyesweb.low_level import BoundingBoxFilledArea, EllipsoidSphericity, PointsDensity

sw_pose = SlidingWindow(max_length=1, n_signals=N_joints, n_dims=3)
sw_pose.append(pos_tensor[100, :, :])

ci  = BoundingBoxFilledArea().compute(pos_tensor[100, :, :])
sph = EllipsoidSphericity().compute(pos_tensor[100, :, :])
pd_ = PointsDensity().compute(pos_tensor[100, :, :])

print(f"Contraction Index : {ci.contraction_index:.4f}")
print(f"Sphericity        : {sph.sphericity:.4f}")
print(f"Points Density    : {pd_.points_density:.4f} mm")

Contraction Index : 1299507.3494
Sphericity        : 0.1705
Points Density    : 485.6579 mm


### 3.4 `SuddennessResult`

In [18]:
from pyeyesweb.mid_level import Suddenness

hand_idx = marker_names.index("LWristOut")
sw_hand = SlidingWindow(max_length=120, n_signals=1, n_dims=3)
sdn_ft  = Suddenness()

for t in range(120):
    sw_hand.append(pos_tensor[t, hand_idx:hand_idx+1, :])

result = sdn_ft(sw_hand)
print(f"is_sudden : {result.is_sudden}")
print(f"alpha     : {result.alpha:.3f}  (stable distribution tail weight)")
print(f"beta      : {result.beta:.3f}   (skewness)")
print(f"gamma     : {result.gamma:.3f}  (scale)")

is_sudden : True
alpha     : 1.436  (stable distribution tail weight)
beta      : 1.126   (skewness)
gamma     : 1.840  (scale)


### 3.5 `RarityResult`

In [19]:
from pyeyesweb.analysis_primitives import Rarity

# Push total kinetic energy values into a longer-horizon window
sw_energy_hist = SlidingWindow(max_length=100, n_signals=1, n_dims=1)
rarity_ft = Rarity(alpha=0.5)
ef2 = KineticEnergy(weights=1.0)
sw_vel = SlidingWindow(max_length=1, n_signals=N_joints, n_dims=3)

for t in range(300):
    sw_vel.append(vel_tensor[t, :, :])
    e_val = ef2(sw_vel).total_energy
    sw_energy_hist.append(e_val)

result = rarity_ft(sw_energy_hist)
print(f"Rarity score : {result.rarity:.4f}")
print("(0 = exactly average; higher = unusually high or low energy moment)")

Rarity score : 0.1650
(0 = exactly average; higher = unusually high or low energy moment)


## 4. `to_flat_dict()` — Flattening for Data Logging

Nested dataclasses are beautiful for code, but problematic for DataFrames and CSV files. `to_flat_dict()` recursively flattens the result into a `{str: scalar}` dictionary.

In [20]:
# KineticEnergy has nested lists and dicts — to_flat_dict unpacks them
result = ef.compute(vel_tensor[100, :, :])
flat = result.to_flat_dict()
print(list(flat.keys())[:8])
# → ['is_valid', 'total_energy', 'energy_x', 'energy_y', 'energy_z',
#    'joint_PELVIS_total', 'joint_PELVIS_x', 'joint_PELVIS_y', ...]

['is_valid', 'total_energy', 'energy_x', 'energy_y', 'energy_z', 'joint_HeadFront_total', 'joint_HeadFront_x', 'joint_HeadFront_y']


### 4.1 The `prefix` argument — namespacing features

When logging multiple features into a single dictionary per frame, `prefix` prevents key collisions:

In [21]:
e_flat  = ef.compute(vel_tensor[100, :, :]).to_flat_dict(prefix="energy")
ci_flat = BoundingBoxFilledArea().compute(pos_tensor[100, :, :]).to_flat_dict(prefix="contraction")

combined = {**e_flat, **ci_flat}
print(list(combined.keys())[:5])
# → ['energy_is_valid', 'energy_total_energy', 'energy_energy_x', ..., 'contraction_is_valid', ...]

['energy_is_valid', 'energy_total_energy', 'energy_energy_x', 'energy_energy_y', 'energy_energy_z']


## 5. Building a Full-Trial Feature Log → Pandas DataFrame

This is the canonical offline analysis pattern: iterate over all frames, extract features, accumulate flat dictionaries, and build a DataFrame in one shot.

In [27]:
ef      = KineticEnergy(weights=1.0, labels=marker_names)
ci_ft   = PointsDensity()
sf      = Smoothness(rate_hz=100.0, metrics=["sparc", "jerk_rms"])

sw_vel  = SlidingWindow(max_length=1,  n_signals=N_joints, n_dims=3)
sw_pos  = SlidingWindow(max_length=1,  n_signals=N_joints, n_dims=3)
sw_spd  = SlidingWindow(max_length=60, n_signals=1,       n_dims=1)

hand_idx = marker_names.index("LWristOut")
hand_speed_arr = np.linalg.norm(vel_tensor[:, hand_idx, :], axis=1)

data_log = []

for t in tqdm(range(N_frames), desc="Building feature log"):
    sw_vel.append(vel_tensor[t, :, :])
    sw_pos.append(pos_tensor[t, :, :])
    sw_spd.append(hand_speed_arr[t])

    if not sw_spd.is_full:
        continue

    row = {"frame": t}
    row.update(ef(sw_vel).to_flat_dict(prefix="energy"))
    row.update(ci_ft(sw_pos).to_flat_dict(prefix="contraction"))
    row.update(sf(sw_spd).to_flat_dict(prefix="smooth"))
    data_log.append(row)

df = pd.DataFrame(data_log)
print(df.shape)
df.head()  # [["frame", "energy_total_energy", "contraction_points_index", "smooth_sparc"]]

Building feature log:   0%|          | 0/6002 [00:00<?, ?it/s]

(5943, 175)


,frame,energy_is_valid,energy_total_energy,energy_energy_x,energy_energy_y,energy_energy_z,energy_joint_HeadFront_total,energy_joint_HeadFront_x,energy_joint_HeadFront_y,energy_joint_HeadFront_z,energy_joint_HeadL_total,energy_joint_HeadL_x,energy_joint_HeadL_y,energy_joint_HeadL_z,energy_joint_HeadR_total,energy_joint_HeadR_x,energy_joint_HeadR_y,energy_joint_HeadR_z,energy_joint_Chest_total,energy_joint_Chest_x,energy_joint_Chest_y,energy_joint_Chest_z,energy_joint_LShoulderTop_total,energy_joint_LShoulderTop_x,energy_joint_LShoulderTop_y,energy_joint_LShoulderTop_z,energy_joint_RShoulderTop_total,energy_joint_RShoulderTop_x,energy_joint_RShoulderTop_y,energy_joint_RShoulderTop_z,energy_joint_SpineThoracic2_total,energy_joint_SpineThoracic2_x,energy_joint_SpineThoracic2_y,energy_joint_SpineThoracic2_z,energy_joint_SpineThoracic12_total,energy_joint_SpineThoracic12_x,energy_joint_SpineThoracic12_y,energy_joint_SpineThoracic12_z,energy_joint_WaistBack_total,energy_joint_WaistBack_x,...,energy_joint_RShinFrontHigh_x,energy_joint_RShinFrontHigh_y,energy_joint_RShinFrontHigh_z,energy_joint_LHeelBack_total,energy_joint_LHeelBack_x,energy_joint_LHeelBack_y,energy_joint_LHeelBack_z,energy_joint_RHeelBack_total,energy_joint_RHeelBack_x,energy_joint_RHeelBack_y,energy_joint_RHeelBack_z,energy_joint_LAnkleOut_total,energy_joint_LAnkleOut_x,energy_joint_LAnkleOut_y,energy_joint_LAnkleOut_z,energy_joint_RAnkleOut_total,energy_joint_RAnkleOut_x,energy_joint_RAnkleOut_y,energy_joint_RAnkleOut_z,energy_joint_LForefoot5_total,energy_joint_LForefoot5_x,energy_joint_LForefoot5_y,energy_joint_LForefoot5_z,energy_joint_RForefoot5_total,energy_joint_RForefoot5_x,energy_joint_RForefoot5_y,energy_joint_RForefoot5_z,energy_joint_LForefoot2_total,energy_joint_LForefoot2_x,energy_joint_LForefoot2_y,energy_joint_LForefoot2_z,energy_joint_RForefoot2_total,energy_joint_RForefoot2_x,energy_joint_RForefoot2_y,energy_joint_RForefoot2_z,contraction_is_valid,contraction_points_density,smooth_is_valid,smooth_sparc,smooth_jerk_rms
0,59,True,17845.810130,8922.015094,3130.919973,5792.875062,493.350022,105.12500,165.620014,222.605008,104.584998,52.019998,26.645001,25.919999,498.745022,98.000000,3.125,397.620022,23.765001,0.405000,2.880000,20.480001,121.544999,19.845001,87.119997,14.580001,208.700008,30.420001,137.780006,40.500000,32.910002,16.820001,9.245001,6.845,155.085005,78.125000,9.680000,67.280004,203.770004,146.205007,...,44.179996,0.605,0.500,5.070,0.845,3.380,0.845,59.130002,32.805003,0.405,25.919999,13.855,7.605,6.125,0.125,11.444999,11.044999,0.320,0.080,1.335,0.605,0.125,0.605,4.950,3.125,1.805,0.020,0.805,0.08,0.005,0.720,5.545,5.120,0.405,0.020,True,485.747375,True,-2.605371,21949.074219
1,60,True,18033.665059,8724.070085,2621.609963,6687.985010,1521.405084,541.20505,3.380000,976.820034,115.449997,93.844997,15.125000,6.480000,417.505005,89.779995,0.045,327.680010,44.845002,0.320000,1.280000,43.245002,129.059999,28.879999,84.500000,15.679999,179.864996,19.219999,115.519997,45.125000,5.335000,1.805000,0.405000,3.125,80.019995,42.319998,2.420000,35.279997,141.640002,69.620002,...,60.500000,0.245,2.205,5.165,2.880,2.205,0.080,37.945000,24.500000,5.445,8.000000,6.190,2.420,2.645,1.125,7.245000,2.880000,3.645,0.720,0.325,0.020,0.125,0.180,10.745,8.000,1.125,1.620,0.445,0.02,0.405,0.020,4.090,2.645,1.125,0.320,True,485.754242,True,-2.585690,10098.830078
2,61,True,16123.529708,7877.889823,2807.804939,5437.834946,241.884993,0.60500,72.000000,169.279993,106.974996,96.604995,9.245001,1.125000,442.069977,167.444986,0.845,273.779991,31.415000,14.045001,1.125000,16.244999,138.494998,57.244998,66.125000,15.125000,168.590008,32.805003,114.005006,21.779999,84.564998,74.419998,10.125000,0.020,119.310006,27.380001,6.125000,85.805005,88.255004,51.005004,...,35.279997,6.125,0.405,1.445,0.720,0.320,0.405,30.105002,12.005000,1.280,16.820001,6.385,4.205,2.000,0.180,30.329999,25.204999,0.005,5.120,1.050,0.080,0.125,0.845,3.870,1.445,2.420,0.005,2.385,0.50,0.080,1.805,5.105,4

**Expected output (values will vary):**

```
(1472, 74)   ← 1472 frames after warm-up, 74 columns

   frame  energy_total_energy  contraction_contraction_index  smooth_sparc
0     60               0.4103                         0.6721        -2.341
1     61               0.3981                         0.6734        -2.388
2     62               0.4210                         0.6699        -2.302
...
```

## 6. Exporting the Feature Matrix to CSV

In [16]:
output_path = "trial0001_features.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} frames × {len(df.columns)} features to {output_path}")

Saved 3553 frames × 175 features to trial0001_features.csv


This CSV can be directly imported into R, MATLAB, or any ML framework (scikit-learn, PyTorch, etc.) for statistical analysis or model training.

## 7. 🧪 Experiment: Filtering Invalid Rows

Not every frame produces valid results. For example, at trial boundaries or during sensor drop-outs, `is_valid=False` rows will appear. Practice filtering them:

In [28]:
# Keep only rows where ALL features computed successfully
valid_mask = (
    (df["energy_is_valid"]) &
    (df["contraction_is_valid"]) &
    (df["smooth_is_valid"])
)
df_clean = df[valid_mask].copy()
print(f"Valid frames: {len(df_clean)} / {len(df)}")

Valid frames: 5943 / 5943


**Now try the following experiments:**
1. Add `Suddenness` to the log. Which joints / trials produce `is_sudden=True`?
2. Add `Rarity` on the energy signal (step: compute energy → push to a long-horizon window → compute rarity). What does a high rarity score correspond to visually?
3. Export three trials into separate CSVs and concatenate them with a `trial_id` column for between-trial comparison.

---

## Summary

| Result class | `is_valid` | Key fields | Nested content? |
|-------------|-----------|-----------|-----------------|
| `FeatureResult` | ✓ | — | No |
| `KineticEnergyResult` | ✓ | `total_energy`, `component_energy`, `joints` | Yes — `to_flat_dict` unpacks |
| `SmoothnessResult` | ✓ | `sparc`, `jerk_rms` | No |
| `ContractionExpansionResult` | ✓ | `contraction_index`, `sphericity`, `points_density` | No |
| `SuddennessResult` | ✓ | `is_sudden`, `alpha`, `beta`, `gamma` | No |
| `RarityResult` | ✓ | `rarity` | No |

In **Notebook 5**, we step back and explore the *scientific* rationale behind the feature hierarchy — the four-layer theoretical framework that gives each of these measurements its meaning.
